# 04. Evaluación: Árbol de Decisión sobre las representaciones

Entrenamiento y evaluación del Árbol de Decisión con las 4 representaciones:
R1 Z-score, R2 binning clínico, R2b binning por KMeans y R3 binning + índice
de atipia. Comparación con 5-fold CV sobre train y test independiente.

In [1]:
import sys
from pathlib import Path

# Permite importar el paquete sin instalarlo (equivalente a `pip install -e .`)
sys.path.insert(0, str(Path('..').resolve()))
from wisconsin_cancer_analysis.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR
from wisconsin_cancer_analysis.modeling.train import (
    REP3_NAME,
    evaluate_representations,
    prepare_train_test,
)
from wisconsin_cancer_analysis.plots import (
    compute_importances,
    plot_arbol,
    plot_confusion_matrix,
    plot_cv_vs_test,
    plot_importancias,
    plot_metricas_comparacion,
)


2026-11-03 18:53:28.828 | INFO     | wisconsin_cancer_analysis.config:<module>:11 - PROJ_ROOT path is: C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis


In [2]:
# Split 80/20 estratificado. X conserva los NaN: la imputacion por mediana
# ocurre dentro del pipeline de cada pliegue (sin fuga de datos).
X_train, X_test, y_train, y_test = prepare_train_test()

pipelines, df_cv, df_test, raw_cv = evaluate_representations(
    X_train, X_test, y_train, y_test
)

df_cv.to_csv(REPORTS_DIR / 'metricas_cv.csv')
df_test.to_csv(REPORTS_DIR / 'metricas_test.csv')

print('--- 5-Fold CV sobre TRAIN ---')
display(df_cv)
print('--- TEST INDEPENDIENTE ---')
display(df_test)

2026-11-03 18:53:37.047 | INFO     | wisconsin_cancer_analysis.modeling.train:prepare_train_test:73 - Train: 559 | Test: 140


2026-11-03 18:53:38.532 | INFO     | wisconsin_cancer_analysis.modeling.train:evaluate_representations:103 - 5-Fold CV sobre TRAIN:
          Rep. 1: Z-Score (9 vars) Rep. 2: Binning (9 vars) Rep. 2b: Binning KMeans (9 vars) Rep. 3: Binning Agrupado (7 vars)
Accuracy           0.9445 ± 0.0154          0.9481 ± 0.0104                  0.9446 ± 0.0103                   0.9660 ± 0.0118
Precision                   0.8949                   0.9191                           0.9288                            0.9540
Recall                      0.9532                   0.9328                           0.9121                            0.9482
F1                          0.9226                   0.9255                           0.9188                            0.9504
2026-11-03 18:53:38.535 | INFO     | wisconsin_cancer_analysis.modeling.train:evaluate_representations:104 - TEST independiente:
          Rep. 1: Z-Score (9 vars) Rep. 2: Binning (9 vars) Rep. 2b: Binning KMeans (9 vars) Rep. 3: Bin

,Rep. 1: Z-Score (9 vars),Rep. 2: Binning (9 vars),Rep. 2b: Binning KMeans (9 vars),Rep. 3: Binning Agrupado (7 vars)
Accuracy,0.9445 ± 0.0154,0.9481 ± 0.0104,0.9446 ± 0.0103,0.9660 ± 0.0118
Precision,0.8949,0.9191,0.9288,0.9540
Recall,0.9532,0.9328,0.9121,0.9482
F1,0.9226,0.9255,0.9188,0.9504


--- TEST INDEPENDIENTE ---


,Rep. 1: Z-Score (9 vars),Rep. 2: Binning (9 vars),Rep. 2b: Binning KMeans (9 vars),Rep. 3: Binning Agrupado (7 vars)
Accuracy,0.9286,0.9357,0.9571,0.9500
Precision,0.9318,0.9333,0.9200,0.9362
Recall,0.8542,0.8750,0.9583,0.9167
F1,0.8913,0.9032,0.9388,0.9263


In [3]:
# Figuras del reporte: comparacion de metricas, CV vs test, matriz de
# confusion y arbol de la Rep. 3
plot_metricas_comparacion(df_test)
plot_cv_vs_test(raw_cv, df_test)
plot_confusion_matrix(pipelines[REP3_NAME], X_test, y_test)
plot_arbol(pipelines[REP3_NAME])

2026-11-03 18:53:39.801 | SUCCESS  | wisconsin_cancer_analysis.plots:plot_metricas_comparacion:139 - Figura -> C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\reports\figures\metricas_comparacion.png


2026-11-03 18:53:41.555 | SUCCESS  | wisconsin_cancer_analysis.plots:plot_cv_vs_test:383 - Figura -> C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\reports\figures\cv_vs_test.png


2026-11-03 18:53:42.253 | SUCCESS  | wisconsin_cancer_analysis.plots:plot_confusion_matrix:350 - Figura -> C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\reports\figures\confusion_matrix_rep3.png


2026-11-03 18:53:46.860 | SUCCESS  | wisconsin_cancer_analysis.plots:plot_arbol:164 - Figura -> C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\reports\figures\arbol_decision_atipia_agrupada.png


In [4]:
# Importancia de caracteristicas: Gini (train) vs permutacion (test)
pipe_rep3 = pipelines[REP3_NAME]
df_imp = compute_importances(pipe_rep3, X_test, y_test)
plot_importancias(df_imp)
df_imp.round(4)


2026-11-03 18:53:51.354 | SUCCESS  | wisconsin_cancer_analysis.plots:plot_importancias:201 - Figura -> C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\reports\figures\importancia_features_rep3.png


,Importancia Gini (train),"Importancia permutacion (test, dF1)"
Atipia_Celular,0.9235,0.0439
Bland_chromatin,0.0205,0.0258
Marginal_adhesion,0.0196,0.0729
Clump_thickness,0.0177,0.0084
Normal_nucleoli,0.0122,-0.0026
Single_epithelial_cell_size,0.0064,-0.0034
Mitoses,0.0000,0.0000


## 1. Importancia Gini (por impureza)

**Idea simple:** un árbol hace preguntas del tipo "¿Atipia_Celular ≤ 1?" para separar benignos de malignos. Esta importancia mide cuánto ayuda cada variable a que los grupos queden más "limpios", es decir, con una sola clase.

**Cómo se calcula:**
1. Cada nodo tiene una **impureza Gini**: `1 − (p_benigno² + p_maligno²)`. Vale 0 si el nodo es puro (todos de una clase) y 0.5 si está mezclado 50/50.
2. Cada vez que el árbol divide un nodo con una variable, calcula cuánto bajó la impureza. Esa reducción se multiplica por la fracción de muestras que pasaron por ese nodo.
3. Se suman todas las reducciones de cada variable, en todos los nodos donde se usó.
4. Se normaliza para que todas sumen 1.

**Qué nos dice:** qué variables usó el árbol y cuánto le sirvieron **durante el entrenamiento**. En nuestro caso, `Atipia_Celular = 0.92` significa que ella sola produjo el 92% de toda la limpieza lograda.

**Limitaciones:**
- Se calcula con los datos de train, así que no dice si la variable sirve con datos nuevos.
- Si dos variables son parecidas, el árbol elige una y la otra queda con poca importancia.
- Favorece variables con muchos valores distintos.

## 2. Importancia por permutación (en test)

**Idea simple:** ¿qué pasa si dañamos una variable? Se baraja al azar la columna de esa variable en el test, de modo que pierda su relación con la respuesta. Si el modelo empeora mucho, esa variable era importante.

**Cómo se calcula:**
1. Se mide el F1 del modelo en el test sin modificar (la línea base).
2. Se baraja una columna y se vuelve a medir el F1.
3. Importancia = F1 original − F1 con la columna barajada.
4. Se repite 30 veces (`n_repeats=30`) y se promedia, porque cada barajado da un resultado distinto.

**Qué nos dice:** cuánto depende el modelo de esa variable al predecir **datos que no vio**. Se interpreta así:
- **0.07** (`Marginal_adhesion`): el F1 cae unos 7 puntos al barajarla. Es una variable valiosa.
- **≈ 0** (`Mitoses`): el modelo no la usa o no le aporta.
- **Negativo** (`Normal_nucleoli`): barajarla no empeora nada, incluso mejora un poco por azar. Es ruido.
